# 09 — Feature Analysis & Single-Signal Baselines

Bu aşama **development / exploratory** analizidir; final test sonucu değildir.

İki scope kullanılır:

- `CLEAN_DEV`: `exclude_from_final_test == 0` → model-selection sanity seti dışarıda
- `FULL_DEV`: mevcut veri setindeki tüm sorular → sensitivity/exploratory

Bu notebook:

- prevalence baseline
- correct vs incorrect feature özetleri
- single-signal AUROC / AUPRC
- bootstrap %95 CI
- raw probability-like score'lar için Brier / NLL / 5-bin exploratory ECE
- CORE Spearman redundancy

hesaplar.

Calibrator eğitimi yapılmaz; o Stage 10'da olacaktır.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install "scikit-learn>=1.5,<1.9" "scipy>=1.13,<2"


Mounted at /content/drive


In [2]:
import hashlib, json
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")
VALIDATED_DATASET_VERSION = "validated_v007"

ASSEMBLED_DIR = PROJECT_ROOT / "features" / "assembled" / VALIDATED_DATASET_VERSION
RESULTS_DIR = PROJECT_ROOT / "results" / "stage09"

STAGE08_CHOICES = (
    {
        "version": "feature_assembly_v2_repair1",
        "manifest": "stage08_manifest_v2_repair1.json",
        "core": "core_model_matrix_v2_repair1.parquet",
        "extended": "extended_model_matrix_v2_repair1.parquet",
        "uses_repair": True,
    },
    {
        "version": "feature_assembly_v1",
        "manifest": "stage08_manifest_v1.json",
        "core": "core_model_matrix_v1.parquet",
        "extended": "extended_model_matrix_v1.parquet",
        "uses_repair": False,
    },
)

SCHEMA_PATH = ASSEMBLED_DIR / "feature_schema_v1.json"

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Repair mevcutsa yalnızca onu içeren dondurulmuş Stage08 v2 seçilir.
# Eski v1'e yalnızca v2/repair mevcut değilse geçilir.
P05_REPAIR = (
    PROJECT_ROOT / "results" / "stage05"
    / f"{VALIDATED_DATASET_VERSION}_stage05_self_confidence_repair_v1.json"
)
v2_manifest_path = ASSEMBLED_DIR / STAGE08_CHOICES[0]["manifest"]

if P05_REPAIR.exists() and not v2_manifest_path.exists():
    raise RuntimeError(
        "Stage05 repair bulundu ancak Stage08 v2 manifesti yok. "
        "Önce repair-aware Stage08'i tamamlayın; eski v1'e geçilmeyecek."
    )

choice = next(
    (candidate for candidate in STAGE08_CHOICES
     if (ASSEMBLED_DIR / candidate["manifest"]).exists()),
    None,
)
if choice is None:
    raise FileNotFoundError(
        f"Stage08 v1/v2 manifesti bulunamadı: {ASSEMBLED_DIR}. "
        "Önce Stage08'i eksiksiz tamamlayın."
    )

M08_PATH = ASSEMBLED_DIR / choice["manifest"]
CORE_PATH = ASSEMBLED_DIR / choice["core"]
EXT_PATH = ASSEMBLED_DIR / choice["extended"]

for p in [M08_PATH, SCHEMA_PATH, CORE_PATH, EXT_PATH]:
    if not p.is_file():
        raise FileNotFoundError(f"Stage08 artifact eksik: {p}")

m08 = load_json(M08_PATH)
schema = load_json(SCHEMA_PATH)

if m08.get("dataset_version") != VALIDATED_DATASET_VERSION:
    raise RuntimeError("Stage08 dataset version mismatch")
if m08.get("assembly_version") != choice["version"]:
    raise RuntimeError("Stage08 assembly version mismatch")
if schema.get("dataset_version") != VALIDATED_DATASET_VERSION:
    raise RuntimeError("Feature schema dataset version mismatch")
if schema.get("assembly_version") != choice["version"]:
    raise RuntimeError("Feature schema assembly version mismatch")

outputs = m08["outputs"]
for label, selected_path in [
    ("core_model_matrix", CORE_PATH),
    ("extended_model_matrix", EXT_PATH),
    ("feature_schema", SCHEMA_PATH),
]:
    recorded_path = Path(outputs[label])
    if recorded_path != selected_path:
        raise RuntimeError(
            f"{label} path mismatch: manifest={recorded_path}, selected={selected_path}"
        )

if file_sha256(CORE_PATH) != outputs["core_model_matrix_sha256"]:
    raise RuntimeError("Core matrix SHA mismatch")
if file_sha256(EXT_PATH) != outputs["extended_model_matrix_sha256"]:
    raise RuntimeError("Extended matrix SHA mismatch")
if schema["schema_sha256"] != m08["feature_schema_sha256"]:
    raise RuntimeError("Feature schema SHA mismatch")
if file_sha256(SCHEMA_PATH) != outputs["feature_schema_file_sha256"]:
    raise RuntimeError("Feature schema file SHA mismatch")

REPAIR_FILE_SHA256 = None
if choice["uses_repair"]:
    if not P05_REPAIR.is_file():
        raise RuntimeError("Stage08 v2 mevcut ama Stage05 repair manifesti eksik")
    repair = load_json(P05_REPAIR)
    upstream = m08.get("upstream", {})
    if (
        file_sha256(P05_REPAIR)
        != upstream.get("stage05_self_confidence_repair_manifest_sha256")
    ):
        raise RuntimeError("Stage05 repair manifest SHA mismatch")
    if (
        repair.get("base_stage05_completion_sha256")
        != upstream.get("stage05_completion_sha256")
    ):
        raise RuntimeError("Stage05 frozen completion SHA mismatch")
    repair_file = Path(repair["repair_file"])
    if not repair_file.is_file():
        raise FileNotFoundError(f"Stage05 repair sidecar eksik: {repair_file}")
    REPAIR_FILE_SHA256 = file_sha256(repair_file)
    if (
        REPAIR_FILE_SHA256 != repair.get("repair_file_sha256")
        or REPAIR_FILE_SHA256
        != upstream.get("stage05_self_confidence_repair_file_sha256")
    ):
        raise RuntimeError("Stage05 repair sidecar SHA mismatch")

core_df = pd.read_parquet(CORE_PATH)
extended_df = pd.read_parquet(EXT_PATH)
CORE_FEATURES = schema["core_features"]
EXTENDED_FEATURES = schema["extended_features"]
TARGET = schema["target"]

# Eksik confidence/diğer predictor'lar varsa analiz başlamadan dur.
for name, frame, features in [
    ("Core", core_df, CORE_FEATURES),
    ("Extended", extended_df, EXTENDED_FEATURES),
]:
    if frame.empty:
        raise RuntimeError(f"{name} matrix boş")
    if frame.duplicated(["question_id", "model_id"]).any():
        raise RuntimeError(f"{name} matrix duplicate question/model key")
    missing_columns = set(features + [TARGET]) - set(frame.columns)
    if missing_columns:
        raise RuntimeError(f"{name} matrix eksik sütunlar: {missing_columns}")
    for feature in features:
        values = pd.to_numeric(frame[feature], errors="coerce").to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise RuntimeError(
                f"{name} matrix: {feature} eksik/nonfinite. "
                "Stage08'i onarın; eksik confidence ile analize devam edilmeyecek."
            )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Seçilen Stage08:", choice["version"])
print("Stage05 repair kullanıldı:", choice["uses_repair"])
print("Sonuç klasörü:", RESULTS_DIR)
print("Core shape:", core_df.shape)
print("Extended shape:", extended_df.shape)
print("Stage08 integrity: PASS")


Seçilen Stage08: feature_assembly_v2_repair1
Stage05 repair kullanıldı: True
Sonuç klasörü: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage09
Core shape: (3000, 18)
Extended shape: (3000, 37)
Stage08 integrity: PASS


In [3]:
FULL_DEV = extended_df.copy()

CLEAN_DEV = extended_df.loc[
    extended_df["exclude_from_final_test"] == 0
].copy()

scope_rows = []

for scope_name, df in [("CLEAN_DEV", CLEAN_DEV), ("FULL_DEV", FULL_DEV)]:
    for model_id, mdf in df.groupby("model_id"):
        y = mdf[TARGET].astype(int)
        scope_rows.append({
            "scope": scope_name,
            "model_id": model_id,
            "n": len(mdf),
            "n_correct": int(y.sum()),
            "n_incorrect": int((y == 0).sum()),
            "accuracy": float(y.mean()),
        })

scope_df = pd.DataFrame(scope_rows)
display(scope_df)

print("FULL_DEV counts by model:")
display(FULL_DEV.groupby("model_id").size().rename("n"))
print("CLEAN_DEV counts by model:")
display(CLEAN_DEV.groupby("model_id").size().rename("n"))


,scope,model_id,n,n_correct,n_incorrect,accuracy
0,CLEAN_DEV,Qwen/Qwen3-4B,970,455,515,0.469072
1,CLEAN_DEV,google/gemma-3-4b-it,970,639,331,0.658763
2,CLEAN_DEV,microsoft/Phi-4-mini-instruct,970,329,641,0.339175
3,FULL_DEV,Qwen/Qwen3-4B,1000,473,527,0.473000
4,FULL_DEV,google/gemma-3-4b-it,1000,666,334,0.666000
5,FULL_DEV,microsoft/Phi-4-mini-instruct,1000,344,656,0.344000


FULL_DEV counts by model:


,n
model_id,
Qwen/Qwen3-4B,1000
google/gemma-3-4b-it,1000
microsoft/Phi-4-mini-instruct,1000


CLEAN_DEV counts by model:


,n
model_id,
Qwen/Qwen3-4B,970
google/gemma-3-4b-it,970
microsoft/Phi-4-mini-instruct,970


## Signal orientation contract

AUROC/AUPRC için yön **etiketlere bakmadan** önceden tanımlanır.

`+1`: yüksek skor = daha yüksek correctness confidence  
`-1`: düşük skor = daha yüksek correctness confidence

`output_content_token_count` monotonic confidence sinyali olarak yorumlanmadığı için single-signal leaderboard'a alınmaz.


In [4]:
SIGNAL_DIRECTION = {
    "mean_logprob": +1,
    "min_logprob": +1,
    "mean_token_entropy_normalized": -1,
    "self_confidence_scaled": +1,
    "verification_p_correct": +1,
    "self_consistency": +1,
    "unknown_response": -1,

    "first_token_logprob": +1,
    "max_token_entropy_normalized": -1,
    "first_token_entropy_normalized": -1,

    "verification_p_uncertain": -1,
    "verification_margin": +1,
    "verification_entropy_normalized": -1,

    "unique_answer_ratio": -1,
    "sample_answer_entropy_normalized": -1,
    "deterministic_majority_agreement": +1,
    "deterministic_sample_agreement_rate": +1,

    "contains_unknown_marker": -1,
    "mixed_answer_unknown": -1,
    "refusal_indicator": -1,

    "unknown_sample_rate": -1,
    "contains_unknown_sample_rate": -1,
    "mixed_unknown_sample_rate": -1,
    "refusal_sample_rate": -1,

    "truncated_by_max_tokens": -1,
    "sample_truncation_rate": -1,
}

NON_ORIENTED_AUXILIARY = ["output_content_token_count"]

missing = [
    f for f in EXTENDED_FEATURES
    if f not in SIGNAL_DIRECTION and f not in NON_ORIENTED_AUXILIARY
]

if missing:
    raise RuntimeError(f"Direction contract eksik: {missing}")

print("Oriented signals:", len(SIGNAL_DIRECTION))


Oriented signals: 26


In [5]:
BOOTSTRAP_REPS = 2000
BOOTSTRAP_SEED = 20260905
EPS = 1e-6

def bootstrap_ci(y, score, metric_fn, seed):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)

    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(BOOTSTRAP_REPS):
        idx = rng.integers(0, len(y), size=len(y))
        yb = y[idx]
        sb = score[idx]

        if len(np.unique(yb)) < 2:
            continue

        vals.append(float(metric_fn(yb, sb)))

    if not vals:
        return np.nan, np.nan, 0

    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi), len(vals)

def binary_nll(y, p):
    p = np.clip(np.asarray(p, dtype=float), EPS, 1.0 - EPS)
    return float(log_loss(y, p, labels=[0, 1]))

def exploratory_ece(y, p, n_bins=5):
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), 0.0, 1.0)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bins = np.digitize(p, edges[1:-1], right=False)

    ece = 0.0
    rows = []

    for b in range(n_bins):
        mask = bins == b
        n_b = int(mask.sum())

        if n_b == 0:
            continue

        mp = float(p[mask].mean())
        my = float(y[mask].mean())
        gap = abs(mp - my)

        ece += (n_b / len(y)) * gap

        rows.append({
            "bin": b,
            "n": n_b,
            "mean_predicted_score": mp,
            "empirical_accuracy": my,
            "abs_gap": float(gap),
        })

    return float(ece), rows


In [6]:
# Prevalence baselines
rows = []

for scope_name, df in [("CLEAN_DEV", CLEAN_DEV), ("FULL_DEV", FULL_DEV)]:
    for model_id, mdf in df.groupby("model_id"):
        y = mdf[TARGET].astype(int).to_numpy()
        p0 = float(y.mean())
        const = np.full(len(y), p0)

        rows.append({
            "scope": scope_name,
            "model_id": model_id,
            "n": len(y),
            "n_correct": int(y.sum()),
            "n_incorrect": int((y == 0).sum()),
            "prevalence_accuracy": p0,
            "auprc_random_baseline": p0,
            "constant_prevalence_brier": float(brier_score_loss(y, const)),
            "constant_prevalence_nll": binary_nll(y, const),
        })

prevalence_df = pd.DataFrame(rows)
display(prevalence_df)

PREVALENCE_PATH = RESULTS_DIR / f"{VALIDATED_DATASET_VERSION}_prevalence_baselines.csv"
prevalence_df.to_csv(PREVALENCE_PATH, index=False)


,scope,model_id,n,n_correct,n_incorrect,prevalence_accuracy,auprc_random_baseline,constant_prevalence_brier,constant_prevalence_nll
0,CLEAN_DEV,Qwen/Qwen3-4B,970,455,515,0.469072,0.469072,0.249043,0.691233
1,CLEAN_DEV,google/gemma-3-4b-it,970,639,331,0.658763,0.658763,0.224794,0.641853
2,CLEAN_DEV,microsoft/Phi-4-mini-instruct,970,329,641,0.339175,0.339175,0.224135,0.640487
3,FULL_DEV,Qwen/Qwen3-4B,1000,473,527,0.473000,0.473000,0.249271,0.691688
4,FULL_DEV,google/gemma-3-4b-it,1000,666,334,0.666000,0.666000,0.222444,0.636975
5,FULL_DEV,microsoft/Phi-4-mini-instruct,1000,344,656,0.344000,0.344000,0.225664,0.643653


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [7]:
# Correct vs incorrect feature summaries
rows = []

for scope_name, df in [("CLEAN_DEV", CLEAN_DEV), ("FULL_DEV", FULL_DEV)]:
    for model_id, mdf in df.groupby("model_id"):
        for feature in EXTENDED_FEATURES:
            for label in [0, 1]:
                s = pd.to_numeric(
                    mdf.loc[mdf[TARGET] == label, feature],
                    errors="coerce",
                ).dropna()

                rows.append({
                    "scope": scope_name,
                    "model_id": model_id,
                    "feature": feature,
                    "manual_is_correct": label,
                    "n": len(s),
                    "mean": float(s.mean()),
                    "std": float(s.std(ddof=1)) if len(s) > 1 else np.nan,
                    "median": float(s.median()),
                    "q25": float(s.quantile(0.25)),
                    "q75": float(s.quantile(0.75)),
                })

distribution_df = pd.DataFrame(rows)

DISTRIBUTION_PATH = RESULTS_DIR / f"{VALIDATED_DATASET_VERSION}_feature_distributions.csv"
distribution_df.to_csv(DISTRIBUTION_PATH, index=False)


In [8]:
# Single-signal AUROC / AUPRC with bootstrap CI
rows = []

scope_defs = [("CLEAN_DEV", CLEAN_DEV), ("FULL_DEV", FULL_DEV)]

for s_idx, (scope_name, df) in enumerate(scope_defs):
    for m_idx, (model_id, mdf) in enumerate(df.groupby("model_id")):
        y = mdf[TARGET].astype(int).to_numpy()

        for f_idx, (feature, direction) in enumerate(SIGNAL_DIRECTION.items()):
            raw = pd.to_numeric(mdf[feature], errors="coerce").to_numpy(dtype=float)

            if not np.isfinite(raw).all():
                raise RuntimeError(f"Nonfinite: {model_id} / {feature}")

            score = raw * float(direction)

            auroc = float(roc_auc_score(y, score))
            auprc = float(average_precision_score(y, score))

            seed = BOOTSTRAP_SEED + s_idx * 100000 + m_idx * 10000 + f_idx * 10

            roc_lo, roc_hi, roc_valid = bootstrap_ci(
                y, score, roc_auc_score, seed
            )

            pr_lo, pr_hi, pr_valid = bootstrap_ci(
                y, score, average_precision_score, seed + 1
            )

            rows.append({
                "scope": scope_name,
                "model_id": model_id,
                "feature": feature,
                "direction": int(direction),
                "n": len(y),
                "n_correct": int(y.sum()),
                "n_incorrect": int((y == 0).sum()),
                "auroc": auroc,
                "auroc_ci95_low": roc_lo,
                "auroc_ci95_high": roc_hi,
                "auprc": auprc,
                "auprc_ci95_low": pr_lo,
                "auprc_ci95_high": pr_hi,
                "auprc_prevalence_baseline": float(y.mean()),
                "bootstrap_valid_auroc": roc_valid,
                "bootstrap_valid_auprc": pr_valid,
            })

single_signal_df = pd.DataFrame(rows)

SINGLE_SIGNAL_PATH = RESULTS_DIR / f"{VALIDATED_DATASET_VERSION}_single_signal_metrics.csv"
single_signal_df.to_csv(SINGLE_SIGNAL_PATH, index=False)

clean_core = (
    single_signal_df.loc[
        (single_signal_df["scope"] == "CLEAN_DEV")
        & (single_signal_df["feature"].isin(CORE_FEATURES))
    ]
    .sort_values(["model_id", "auroc"], ascending=[True, False])
)

display(
    clean_core[
        [
            "model_id",
            "feature",
            "n",
            "auroc",
            "auroc_ci95_low",
            "auroc_ci95_high",
            "auprc",
            "auprc_prevalence_baseline",
        ]
    ]
)


,model_id,feature,n,auroc,auroc_ci95_low,auroc_ci95_high,auprc,auprc_prevalence_baseline
4,Qwen/Qwen3-4B,verification_p_correct,970,0.898671,0.878708,0.919052,0.893999,0.469072
3,Qwen/Qwen3-4B,self_confidence_scaled,970,0.859441,0.835612,0.882329,0.790005,0.469072
1,Qwen/Qwen3-4B,min_logprob,970,0.773065,0.743166,0.800857,0.740949,0.469072
2,Qwen/Qwen3-4B,mean_token_entropy_normalized,970,0.750940,0.720602,0.782611,0.719540,0.469072
0,Qwen/Qwen3-4B,mean_logprob,970,0.746192,0.716988,0.777160,0.717070,0.469072
5,Qwen/Qwen3-4B,self_consistency,970,0.714540,0.685656,0.742103,0.618160,0.469072
6,Qwen/Qwen3-4B,unknown_response,970,0.587251,0.571288,0.603774,0.516979,0.469072
28,google/gemma-3-4b-it,mean_token_entropy_normalized,970,0.792860,0.764167,0.821691,0.877182,0.658763
27,google/gemma-3-4b-it,min_logprob,970,0.787063,0.756310,0.815496,0.875221,0.658763
26,google/gemma-3-4b-it,mean_logprob,970,0.783891,0.754125,0.812060,0.873825,0.658763


## Raw probability-like scores

Brier/NLL/ECE yalnız `[0,1]` aralığındaki ham behavioral score'lar için hesaplanır:

- `self_confidence_scaled`
- `verification_p_correct`
- `self_consistency`
- `deterministic_sample_agreement_rate`

Bunlar kalibre edilmiş `P(correct)` değildir; ham baseline'dır.


In [9]:
PROBABILITY_LIKE = [
    "self_confidence_scaled",
    "verification_p_correct",
    "self_consistency",
    "deterministic_sample_agreement_rate",
]

metric_rows = []
rel_rows = []

for scope_name, df in [("CLEAN_DEV", CLEAN_DEV), ("FULL_DEV", FULL_DEV)]:
    for model_id, mdf in df.groupby("model_id"):
        y = mdf[TARGET].astype(int).to_numpy()

        for signal in PROBABILITY_LIKE:
            p = pd.to_numeric(mdf[signal], errors="coerce").to_numpy(dtype=float)

            if ((p < -1e-12) | (p > 1 + 1e-12)).any():
                raise RuntimeError(f"{signal} outside [0,1]")

            p = np.clip(p, 0.0, 1.0)
            ece, bins = exploratory_ece(y, p, n_bins=5)

            metric_rows.append({
                "scope": scope_name,
                "model_id": model_id,
                "signal": signal,
                "n": len(y),
                "empirical_accuracy": float(y.mean()),
                "mean_score": float(p.mean()),
                "mean_score_minus_accuracy": float(p.mean() - y.mean()),
                "brier": float(brier_score_loss(y, p)),
                "nll": binary_nll(y, p),
                "ece_5_bins_exploratory": ece,
            })

            for r in bins:
                rel_rows.append({
                    "scope": scope_name,
                    "model_id": model_id,
                    "signal": signal,
                    **r,
                })

probability_df = pd.DataFrame(metric_rows)
reliability_df = pd.DataFrame(rel_rows)

PROBABILITY_PATH = RESULTS_DIR / f"{VALIDATED_DATASET_VERSION}_raw_probability_like_metrics.csv"
RELIABILITY_PATH = RESULTS_DIR / f"{VALIDATED_DATASET_VERSION}_raw_reliability_bins.csv"

probability_df.to_csv(PROBABILITY_PATH, index=False)
reliability_df.to_csv(RELIABILITY_PATH, index=False)

display(
    probability_df.loc[
        probability_df["scope"] == "CLEAN_DEV"
    ].sort_values(["model_id", "brier"])
)


,scope,model_id,signal,n,empirical_accuracy,mean_score,mean_score_minus_accuracy,brier,nll,ece_5_bins_exploratory
1,CLEAN_DEV,Qwen/Qwen3-4B,verification_p_correct,970,0.469072,0.472036,0.002964,0.168877,1.601189,0.168151
0,CLEAN_DEV,Qwen/Qwen3-4B,self_confidence_scaled,970,0.469072,0.751959,0.282887,0.239859,1.498336,0.284474
3,CLEAN_DEV,Qwen/Qwen3-4B,deterministic_sample_agreement_rate,970,0.469072,0.750515,0.281443,0.308577,3.234476,0.297938
2,CLEAN_DEV,Qwen/Qwen3-4B,self_consistency,970,0.469072,0.800412,0.331340,0.321608,3.202430,0.331340
7,CLEAN_DEV,google/gemma-3-4b-it,deterministic_sample_agreement_rate,970,0.658763,0.877320,0.218557,0.252206,2.769156,0.236701
6,CLEAN_DEV,google/gemma-3-4b-it,self_consistency,970,0.658763,0.904330,0.245567,0.253485,2.714338,0.245567
4,CLEAN_DEV,google/gemma-3-4b-it,self_confidence_scaled,970,0.658763,0.906485,0.247722,0.265177,2.410532,0.259309
5,CLEAN_DEV,google/gemma-3-4b-it,verification_p_correct,970,0.658763,0.860009,0.201246,0.274458,2.361044,0.274102
9,CLEAN_DEV,microsoft/Phi-4-mini-instruct,verification_p_correct,970,0.339175,0.582261,0.243085,0.271429,1.090061,0.254178
11,CLEAN_DEV,microsoft/Phi-4-mini-instruct,deterministic_sample_agreement_rate,970,0.339175,0.469897,0.130722,0.288412,2.339793,0.223093


In [10]:
# CORE Spearman redundancy
rows = []

for scope_name, df in [("CLEAN_DEV", CLEAN_DEV), ("FULL_DEV", FULL_DEV)]:
    for model_id, mdf in df.groupby("model_id"):
        for i, a in enumerate(CORE_FEATURES):
            for b in CORE_FEATURES[i + 1:]:
                rho, pval = spearmanr(
                    pd.to_numeric(mdf[a], errors="coerce"),
                    pd.to_numeric(mdf[b], errors="coerce"),
                )

                rows.append({
                    "scope": scope_name,
                    "model_id": model_id,
                    "feature_a": a,
                    "feature_b": b,
                    "spearman_rho": float(rho) if np.isfinite(rho) else np.nan,
                    "p_value_exploratory": float(pval) if np.isfinite(pval) else np.nan,
                })

correlation_df = pd.DataFrame(rows)

CORRELATION_PATH = RESULTS_DIR / f"{VALIDATED_DATASET_VERSION}_core_spearman_correlations.csv"
correlation_df.to_csv(CORRELATION_PATH, index=False)

strong = (
    correlation_df.loc[
        (correlation_df["scope"] == "CLEAN_DEV")
        & (correlation_df["spearman_rho"].abs() >= 0.8)
    ]
    .sort_values(
        "spearman_rho",
        key=lambda s: s.abs(),
        ascending=False,
    )
)

print("CLEAN_DEV |rho| >= 0.8 pairs:", len(strong))
display(strong.head(50))


CLEAN_DEV |rho| >= 0.8 pairs: 10


,scope,model_id,feature_a,feature_b,spearman_rho,p_value_exploratory
29,CLEAN_DEV,google/gemma-3-4b-it,mean_logprob,mean_token_entropy_normalized,-0.993083,0.000000e+00
28,CLEAN_DEV,google/gemma-3-4b-it,mean_logprob,min_logprob,0.983653,0.000000e+00
1,CLEAN_DEV,Qwen/Qwen3-4B,mean_logprob,mean_token_entropy_normalized,-0.981720,0.000000e+00
35,CLEAN_DEV,google/gemma-3-4b-it,min_logprob,mean_token_entropy_normalized,-0.969151,0.000000e+00
57,CLEAN_DEV,microsoft/Phi-4-mini-instruct,mean_logprob,mean_token_entropy_normalized,-0.959891,0.000000e+00
0,CLEAN_DEV,Qwen/Qwen3-4B,mean_logprob,min_logprob,0.943943,0.000000e+00
7,CLEAN_DEV,Qwen/Qwen3-4B,min_logprob,mean_token_entropy_normalized,-0.907524,0.000000e+00
22,CLEAN_DEV,Qwen/Qwen3-4B,self_confidence_scaled,verification_p_correct,0.880004,2.973297e-315
11,CLEAN_DEV,Qwen/Qwen3-4B,min_logprob,self_consistency,0.821360,3.040397e-238
67,CLEAN_DEV,microsoft/Phi-4-mini-instruct,min_logprob,self_consistency,0.801275,3.626789e-218


In [11]:
# CORE leaderboard
rows = []

clean = single_signal_df.loc[
    single_signal_df["scope"] == "CLEAN_DEV"
]

for model_id, mdf in clean.groupby("model_id"):
    eligible = (
        mdf.loc[mdf["feature"].isin(CORE_FEATURES)]
        .sort_values("auroc", ascending=False)
    )

    for rank, (_, r) in enumerate(eligible.iterrows(), start=1):
        rows.append({
            "model_id": model_id,
            "rank": rank,
            "feature": r["feature"],
            "auroc": r["auroc"],
            "auroc_ci95_low": r["auroc_ci95_low"],
            "auroc_ci95_high": r["auroc_ci95_high"],
            "auprc": r["auprc"],
            "prevalence_baseline": r["auprc_prevalence_baseline"],
        })

leaderboard_df = pd.DataFrame(rows)

LEADERBOARD_PATH = RESULTS_DIR / f"{VALIDATED_DATASET_VERSION}_clean_dev_core_leaderboard.csv"
leaderboard_df.to_csv(LEADERBOARD_PATH, index=False)

display(leaderboard_df)


,model_id,rank,feature,auroc,auroc_ci95_low,auroc_ci95_high,auprc,prevalence_baseline
0,Qwen/Qwen3-4B,1,verification_p_correct,0.898671,0.878708,0.919052,0.893999,0.469072
1,Qwen/Qwen3-4B,2,self_confidence_scaled,0.859441,0.835612,0.882329,0.790005,0.469072
2,Qwen/Qwen3-4B,3,min_logprob,0.773065,0.743166,0.800857,0.740949,0.469072
3,Qwen/Qwen3-4B,4,mean_token_entropy_normalized,0.750940,0.720602,0.782611,0.719540,0.469072
4,Qwen/Qwen3-4B,5,mean_logprob,0.746192,0.716988,0.777160,0.717070,0.469072
5,Qwen/Qwen3-4B,6,self_consistency,0.714540,0.685656,0.742103,0.618160,0.469072
6,Qwen/Qwen3-4B,7,unknown_response,0.587251,0.571288,0.603774,0.516979,0.469072
7,google/gemma-3-4b-it,1,mean_token_entropy_normalized,0.792860,0.764167,0.821691,0.877182,0.658763
8,google/gemma-3-4b-it,2,min_logprob,0.787063,0.756310,0.815496,0.875221,0.658763
9,google/gemma-3-4b-it,3,mean_logprob,0.783891,0.754125,0.812060,0.873825,0.658763


In [12]:
# Stage-09 exploratory manifest
MANIFEST_PATH = RESULTS_DIR / f"{VALIDATED_DATASET_VERSION}_stage09_exploratory_manifest.json"

manifest = {
    "stage": "09_feature_analysis_baselines",
    "dataset_version": VALIDATED_DATASET_VERSION,
    "stage08_manifest_sha256": file_sha256(M08_PATH),
    "stage08_assembly_version": choice["version"],
    "stage05_repair_used": choice["uses_repair"],
    "stage05_repair_file_sha256": REPAIR_FILE_SHA256,
    "analysis_status": "development_exploratory_not_final_test",
    "primary_scope": "CLEAN_DEV",
    "primary_scope_definition": "exclude_from_final_test == 0",
    "sensitivity_scope": "FULL_DEV",
    "bootstrap_reps": BOOTSTRAP_REPS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "single_signal_direction_contract": SIGNAL_DIRECTION,
    "non_oriented_auxiliary_features": NON_ORIENTED_AUXILIARY,
    "probability_like_signals": PROBABILITY_LIKE,
    "exploratory_ece_bins": 5,
    "outputs": {
        "prevalence_baselines": str(PREVALENCE_PATH),
        "feature_distributions": str(DISTRIBUTION_PATH),
        "single_signal_metrics": str(SINGLE_SIGNAL_PATH),
        "raw_probability_like_metrics": str(PROBABILITY_PATH),
        "raw_reliability_bins": str(RELIABILITY_PATH),
        "core_spearman_correlations": str(CORRELATION_PATH),
        "clean_dev_core_leaderboard": str(LEADERBOARD_PATH),
    },
    "final_test_used": False,
    "calibrator_trained": False,
}

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        old = json.load(f)

    if old != manifest:
        raise RuntimeError("Existing Stage09 manifest differs")
else:
    with open(MANIFEST_PATH, "x", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

print("STAGE 09 EXPLORATORY ANALYSIS DONDURULDU:")
print(MANIFEST_PATH)


STAGE 09 EXPLORATORY ANALYSIS DONDURULDU:
/content/drive/MyDrive/LLM_Confidence_Thesis/results/stage09/validated_v007_stage09_exploratory_manifest.json


# 09 tamamlanma kriterleri

- [ ] Stage08 matrix SHA doğrulandı
- [ ] CLEAN_DEV / FULL_DEV ayrıldı
- [ ] CLEAN_DEV model-selection overlap'ı dışladı
- [ ] Signal direction contract explicit
- [ ] Prevalence baseline hesaplandı
- [ ] Correct/incorrect feature özetleri üretildi
- [ ] Single-signal AUROC/AUPRC hesaplandı
- [ ] Bootstrap %95 CI üretildi
- [ ] Raw probability-like Brier/NLL/ECE üretildi
- [ ] CORE Spearman redundancy incelendi
- [ ] Stage09 manifest `not_final_test` olarak yazıldı
- [ ] Calibrator eğitilmedi

## Sonraki aşama — 10 Post-hoc Training

Stage10'da model başına OOF/repeated stratified CV ile L2 logistic regression ve single-signal
post-hoc baselines kurulacak. Final dataset freeze olmadan final test oluşturulmayacak.
